# Exploratory Data Analysis — NYC Short-Term Rental Prices

This notebook performs a first-pass exploration of the raw `sample.csv` artifact produced by the `download` step of the pipeline. The goal is to spot obvious data quality issues (missing values, wrong dtypes, outliers) before we encode the fixes into the reusable `basic_cleaning` step.

Run `mlflow run . -P steps=download` from the project root before opening this notebook, then launch it with `mlflow run src/eda`.

## 1. Fetch the raw artifact from Weights & Biases

We open a W&B run with `save_code=True` so this notebook itself is versioned alongside the artifacts it produces.

In [ ]:
import wandb
import pandas as pd

run = wandb.init(project="nyc_airbnb", group="eda", save_code=True)
local_path = wandb.use_artifact("sample.csv:latest").file()
df = pd.read_csv(local_path)

## 2. Profile the raw data

We use `ydata-profiling` (the maintained successor to `pandas-profiling`) to get a quick overview of missingness, distributions, and correlations.

In [ ]:
from ydata_profiling import ProfileReport

profile = ProfileReport(df, title="Raw sample.csv profile")
profile.to_widgets()

### Observations

- `last_review` is stored as a string, not a date.
- Several columns have missing values, most notably `last_review` and `reviews_per_month` (listings with no reviews yet).
- The `price` column has a long right tail with some `$0` listings and some extreme outliers. After discussion with stakeholders, we agreed to keep listings priced between **$10** and **$350** per night.
- A handful of rows have `longitude`/`latitude` values that fall outside the NYC metro area — likely data entry errors.

## 3. Fix the issues found above

Note that we deliberately do **not** impute missing values here — that is left to the inference pipeline (`train_random_forest`) so the same imputation logic is also applied in production at prediction time.

In [ ]:
# Drop price outliers
min_price = 10
max_price = 350
idx = df['price'].between(min_price, max_price)
df = df[idx].copy()

# Convert last_review to datetime
df['last_review'] = pd.to_datetime(df['last_review'])

# Drop rows outside of the proper NYC geographic boundaries
idx = df['longitude'].between(-74.25, -73.50) & df['latitude'].between(40.5, 41.2)
df = df[idx].copy()

## 4. Verify the fix

In [ ]:
df.info()

In [ ]:
df['price'].describe()

`last_review` is now a proper `datetime64` column, the price range is bounded as expected, and the geographic outliers are gone. This is exactly the logic encoded in `src/basic_cleaning/run.py`, so the production pipeline reproduces what we validated here.

## 5. Close out the run

Always call `run.finish()` and then close the notebook via *File &rarr; Close and Halt* and quit Jupyter from the main page (do **not** use Ctrl-C) so the underlying `mlflow run` terminates cleanly.

In [ ]:
run.finish()